# 🚀 Модуль 6. Урок 3. Смешивание данных (Data Blending) и объединение источников

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Enlyl/data-analyst-2027/blob/main/course/module_06_bi_looker_studio/lesson_03_data_blending_joins_in_bi.ipynb)

## 📌 Бизнес-контекст и методология
Data Blending в Looker Studio выполняет LEFT JOIN между источниками по общему ключу (Join Key).
Важно: смешивание происходит ПОСЛЕ группировки, поэтому данные объединяются на агрегированном уровне.

In [1]:
# 🚀 Универсальная инициализация окружения и данных
import os
import urllib.request
import sqlite3
import pandas as pd
import numpy as np

def ensure_dataset(filename, subfolder=""):
    """Автоматически находит датасет локально или скачивает с GitHub репозитория курса."""
    if os.path.exists(filename):
        return filename
    candidates = [
        os.path.join("course", "datasets", subfolder, filename),
        os.path.join("course", "datasets", filename),
        os.path.join("..", "datasets", subfolder, filename),
        os.path.join("..", "datasets", filename),
        os.path.join("..", "..", "course", "datasets", subfolder, filename),
        os.path.join("datasets", subfolder, filename),
        os.path.join("datasets", filename),
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    raw_base = "https://raw.githubusercontent.com/Enlyl/data-analyst-2027/main/course/datasets"
    url = f"{raw_base}/{subfolder}/{filename}" if subfolder else f"{raw_base}/{filename}"
    print(f"📥 Загрузка {filename} из репозитория курса...")
    urllib.request.urlretrieve(url, filename)
    print(f"✅ Датасет {filename} успешно готов!")
    return filename

# Загрузка ключевых датасетов
customers_csv = ensure_dataset("customers.csv", subfolder="raw")
orders_csv = ensure_dataset("orders.csv", subfolder="raw")
saas_csv = ensure_dataset("saas_users.csv", subfolder="raw")
print("🟢 Окружение готово к анализу данных.")


🟢 Окружение готово к анализу данных.


## 🔍 Разбор механики и демонстрация на реальных данных

In [2]:
import pandas as pd

# Таблица 1: Затраты на маркетинг
df_spend = pd.DataFrame({
    'channel': ['Paid Search', 'Social Ads', 'Referral', 'SEO'],
    'ad_spend': [120_000, 85_000, 15_000, 30_000]
})

# Таблица 2: Выручка от привлеченных клиентов
df_rev = pd.DataFrame({
    'channel': ['Paid Search', 'Social Ads', 'Referral', 'Email'],
    'revenue': [340_000, 160_000, 45_000, 70_000]
})

# Blending (Left Join)
blended = pd.merge(df_spend, df_rev, on='channel', how='left').fillna(0)
blended['roas'] = blended['revenue'] / blended['ad_spend']
print(blended)


       channel  ad_spend   revenue      roas
0  Paid Search    120000  340000.0  2.833333
1   Social Ads     85000  160000.0  1.882353
2     Referral     15000   45000.0  3.000000
3          SEO     30000       0.0  0.000000


### 🎯 Практическое задание 1: Суммарный ROAS маркетинга
Рассчитайте совокупный ROAS по всем объединенным каналам.

In [3]:
### ВАШ КОД ЗДЕСЬ
total_roas = blended['revenue'].sum() / blended['ad_spend'].sum()
print(f"Совокупный ROAS: {total_roas:.2f}x")

assert total_roas > 1.0, "Маркетинг в целом окупаем"
print("✅ Тест успешно пройден: смешивание данных и расчет метрики выполнены!")


Совокупный ROAS: 2.18x
✅ Тест успешно пройден: смешивание данных и расчет метрики выполнены!


### 💡 Ограничения Data Blending
При смешивании более 3-4 источников дашборды в BI начинают тормозить. Индустриальная рекомендация: готовить плоскую таблицу в BigQuery / ClickHouse.